In [121]:
from collections import Counter
import argparse
import os
import json

import numpy as np
from pathlib import Path
from tqdm import tqdm
from p_tqdm import p_map
from scipy.stats import wasserstein_distance

from pymatgen.core.structure import Structure
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher
from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
from matminer.featurizers.composition.composite import ElementProperty

In [168]:
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
from pymatgen.io.cif import CifWriter

In [121]:
#added by Tsach


from eval_utils import (
    smact_validity, structure_validity, CompScaler, get_fp_pdist,
    load_config, load_data, get_crystals_list, prop_model_eval, compute_cov)


CrystalNNFP = CrystalNNFingerprint.from_preset("ops")
CompFP = ElementProperty.from_preset('magpie')

Percentiles = {
    'mp20': np.array([-3.17562208, -2.82196882, -2.52814761]),
    'carbon': np.array([-154.527093, -154.45865733, -154.44206825]),
    'perovskite': np.array([0.43924842, 0.61202443, 0.7364607]),
}

COV_Cutoffs = {
    'mp20': {'struc': 0.4, 'comp': 10.},
    'carbon': {'struc': 0.2, 'comp': 4.},
    'perovskite': {'struc': 0.2, 'comp': 4},
}

In [6]:
# parser = argparse.ArgumentParser()
# parser.add_argument('--root_path', default='/home/gridsan/tmackey/hydra/singlerun/2023-11-01/og_CDVAE_neg_mask_mp_20') # changed the default to the 
# parser.add_argument('--label', default='')
# parser.add_argument('--tasks', nargs='+', default=['recon']) #changing the default to recon only 
# parser.add_argument('--compare_diffraction_patterns', default=False) #no diffraction pattern comparison 
# args = parser.parse_args()

In [7]:
all_metrics = {}

cfg = load_config('/home/gridsan/tmackey/hydra/singlerun/2023-11-01/og_CDVAE_neg_mask_mp_20')
eval_model_name = cfg.data.eval_model_name

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


In [10]:
def get_file_paths(root_path, task, label='', suffix='pt'):
    if label == '':
        out_name = f'eval_{task}.{suffix}'
    else:
        out_name = f'eval_{task}_{label}.{suffix}'
    out_name = os.path.join(root_path, out_name)
    return out_name

In [11]:
recon_file_path = get_file_paths('/home/gridsan/tmackey/hydra/singlerun/2023-11-01/og_CDVAE_neg_mask_mp_20', 'recon',"")

In [16]:
# def get_crystal_array_list(file_path, batch_idx=0):
file_path = recon_file_path
data = load_data(file_path)

In [84]:
print(data['eval_setting'])

Namespace(batch_size=500, disable_bar=False, down_sample_traj_step=10, force_atom_types=False, force_num_atoms=True, label='', min_sigma=0, model_path='/home/gridsan/tmackey/hydra/singlerun/2023-11-01/og_CDVAE_neg_mask_mp_20', n_step_each=100, num_batches=3, num_batches_to_samples=20, num_evals=10, save_traj=True, start_from='data', step_lr=0.0001, tasks=['recon'])


In [204]:
print(data['atom_types'][0][range(5)])

tensor([31, 52, 31, 52, 52])


In [205]:
print(data['atom_types'][1][range(5)])

tensor([52, 31, 52, 31, 52])


In [91]:
def get_crystals_list(
        frac_coords, atom_types, lengths, angles, num_atoms):
    """
    args:
        frac_coords: (num_atoms, 3)
        atom_types: (num_atoms)
        lengths: (num_crystals)
        angles: (num_crystals)
        num_atoms: (num_crystals)
    """
    assert frac_coords.size(0) == atom_types.size(0) == num_atoms.sum()
    assert lengths.size(0) == angles.size(0) == num_atoms.size(0)

    start_idx = 0
    crystal_array_list = []
    for batch_idx, num_atom in enumerate(num_atoms.tolist()):
        cur_frac_coords = frac_coords.narrow(0, start_idx, num_atom)
        cur_atom_types = atom_types.narrow(0, start_idx, num_atom)
        cur_lengths = lengths[batch_idx]
        cur_angles = angles[batch_idx]

        crystal_array_list.append({
            'frac_coords': cur_frac_coords.detach().cpu().numpy(),
            'atom_types': cur_atom_types.detach().cpu().numpy(),
            'lengths': cur_lengths.detach().cpu().numpy(),
            'angles': cur_angles.detach().cpu().numpy(),
        })
        start_idx = start_idx + num_atom
    return crystal_array_list


In [106]:
def get_crystal_array_list(file_path, batch_idx=0):
    data = load_data(file_path)
    crys_array_list = get_crystals_list(
        data['frac_coords'][batch_idx],
        data['atom_types'][batch_idx],
        data['lengths'][batch_idx],
        data['angles'][batch_idx],
        data['num_atoms'][batch_idx])

    if 'input_data_batch' in data:
        batch = data['input_data_batch']
        if isinstance(batch, dict):
            true_crystal_array_list = get_crystals_list(
                batch['frac_coords'], batch['atom_types'], batch['lengths'],
                batch['angles'], batch['num_atoms'])
        else:
            true_crystal_array_list = get_crystals_list(
                batch.frac_coords, batch.atom_types, batch.lengths,
                batch.angles, batch.num_atoms)
    else:
        true_crystal_array_list = None

    return crys_array_list, true_crystal_array_list

In [188]:
class Crystal(object):
    def __init__(self, crys_array_dict):
        self.frac_coords = crys_array_dict['frac_coords']
        self.atom_types = crys_array_dict['atom_types']
        self.lengths = crys_array_dict['lengths']
        self.angles = crys_array_dict['angles']
        self.dict = crys_array_dict

        self.get_structure()
        self.get_composition()
        self.get_validity()
        self.get_fingerprints()
    def get_structure(self):
        if min(self.lengths.tolist()) < 0:
            self.constructed = False
            self.invalid_reason = 'non_positive_lattice'
        else:
            try:
                self.structure = Structure(
                    lattice=Lattice.from_parameters(
                        *(self.lengths.tolist() + self.angles.tolist())),
                    species=self.atom_types, coords=self.frac_coords, coords_are_cartesian=False)
                self.constructed = True
            except Exception:
                self.constructed = False
                self.invalid_reason = 'construction_raises_exception'
            if self.structure.volume < 0.1:
                self.constructed = False
                self.invalid_reason = 'unrealistically_small_lattice'
    def get_composition(self):
        elem_counter = Counter(self.atom_types)
        composition = [(elem, elem_counter[elem])
                       for elem in sorted(elem_counter.keys())]
        elems, counts = list(zip(*composition))
        counts = np.array(counts)
        counts = counts / np.gcd.reduce(counts)
        self.elems = elems
        self.comps = tuple(counts.astype('int').tolist())
    def get_validity(self):
        self.comp_valid = smact_validity(self.elems, self.comps)
        if self.constructed:
            self.struct_valid = structure_validity(self.structure)
        else:
            self.struct_valid = False
        self.valid = self.comp_valid and self.struct_valid
    def get_fingerprints(self):
        elem_counter = Counter(self.atom_types)
        comp = Composition(elem_counter)
        self.comp_fp = CompFP.featurize(comp)
        try:
            site_fps = [CrystalNNFP.featurize(
                self.structure, i) for i in range(len(self.structure))]
        except Exception:
            # counts crystal as invalid if fingerprint cannot be constructed.
            print('oops')
            self.valid = False
            self.comp_fp = None
            self.struct_fp = None
            return
        self.struct_fp = np.array(site_fps).mean(axis=0)

In [208]:
class RecEval(object):

    def __init__(self, pred_crys, gt_crys, stol=0.5, angle_tol=10, ltol=0.3): #original values of stol=0.5, angle_tol=10, ltol=0.3
        assert len(pred_crys) == len(gt_crys)
        self.matcher = StructureMatcher(
            stol=stol, angle_tol=angle_tol, ltol=ltol)
        self.preds = pred_crys
        self.gts = gt_crys

    def get_match_rate_and_rms(self):
        def process_one(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                rms_dist = self.matcher.get_rms_dist(
                    pred.structure, gt.structure)
                rms_dist = None if rms_dist is None else rms_dist[0]
                return rms_dist
            except Exception:
                return None
        #define a function that gets the diffraction patterns for pred and gt and returns the RMSD between them
        def process_diff_pattern(pred, gt, is_valid):
            if not is_valid:
                return None
            try:
                #get the structures
                pred_structure = pred.structure
                gt_structure = gt.structure
                pred_pattern = xrd_calculator.get_pattern(pred_structure)
                gt_pattern = xrd_calculator.get_pattern(gt_structure)

                pred_adjusted_vector = np.zeros(256)
                minimum = min(256, len(pred_pattern.x))
                pred_adjusted_vector[:minimum] = pred_pattern.x[:minimum]

                gt_adjusted_vector = np.zeros(256)
                minimum = min(256, len(gt_pattern.x))
                gt_adjusted_vector[:minimum] = gt_pattern.x[:minimum]
                
                #calculate the RMSD between the two patterns
                print(pred_adjusted_vector)
                print(gt_adjusted_vector)
                rms_dist = np.sqrt(np.mean((pred_adjusted_vector - gt_adjusted_vector)**2))

                return rms_dist
            except Exception:
                return None    

        validity = [c.valid for c in self.preds]
        
        print(validity)

        rms_dists = []
        evaluate_diff_pattern = False
        if evaluate_diff_pattern:
            diff_dists = []
        for i in tqdm(range(len(self.preds))):
            rms_dists.append(process_one(
                self.preds[i], self.gts[i], validity[i]))
            if evaluate_diff_pattern:
                diff_dists.append(process_diff_pattern(self.preds[i], self.gts[i], validity[i]))
        rms_dists = np.array(rms_dists)
        if evaluate_diff_pattern:
            diff_dists = np.array(diff_dists)
            average_diff_dist = diff_dists[diff_dists != None].mean()
            #print out all the diff dists
        else:
            average_diff_dist = None
        match_rate = sum(rms_dists != None) / len(self.preds)
        mean_rms_dist = rms_dists[rms_dists != None].mean()

        return {'match_rate': match_rate,
                'rms_dist': mean_rms_dist,
                'diff_dist': average_diff_dist,
                'rmsd_values': rms_dists}

    def get_metrics(self):
        return self.get_match_rate_and_rms()

In [210]:
from multiprocessing import Pool, cpu_count

def create_crystal(x):
    return Crystal(x)
num_cores = cpu_count()
print(num_cores)

80


In [237]:
total_rmsd = []
for eval_num in range(10): 
    
    crys_array_list, true_crystal_array_list = get_crystal_array_list(file_path, batch_idx=eval_num)

    crys_array_list = crys_array_list[0:100]

    pred_crys = []
    counter = 0 
    for x in crys_array_list: 
        pred_crys.append(Crystal(x))

    true_crystal_array_list = true_crystal_array_list[0:100]

    gt_crys = []
    counter = 0 

    for x in true_crystal_array_list: 
        gt_crys.append(Crystal(x))

    rec_evaluator = RecEval(pred_crys, gt_crys)
    recon_metrics = rec_evaluator.get_metrics()
    total_rmsd.append(recon_metrics['rmsd_values'])

[False, True, True, False, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, False, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, False, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True]


100%|██████████| 100/100 [00:00<00:00, 103.79it/s]


[False, True, True, False, True, True, False, True, True, True, True, False, True, True, False, True, False, True, False, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, False, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, False, True, True, False, False, True, True, True, True, True, True]


100%|██████████| 100/100 [00:00<00:00, 100.42it/s]


[False, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, False, False, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, False, True, True, True, True, True, True]


...

100%|██████████| 100/100 [00:00<00:00, 108.35it/s]


[False, True, True, False, True, True, True, True, True, True, True, False, True, True, False, True, True, True, False, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, False, True, True, True, True, True, True]


100%|██████████| 100/100 [00:00<00:00, 100.94it/s]


[False, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, True, True, False, False, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, False, True, True, True, False, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, False, True, False, True, True, True, True, True, True, True, True, False, True, True, True, False, True, True, True, True, True, True, True, True, False, True]


100%|██████████| 100/100 [00:00<00:00, 102.01it/s]


In [238]:
total_rmsd

[array([None, None, None, None, None, None, None, 0.05516386955059857,
        None, None, 0.013162929636230492, None, None, None, None,
        0.012938712541734875, None, None, None, None, 0.016387707610090094,
        None, 0.014483688790807775, 0.018444475652651473, None,
        0.011371860877523692, None, None, None, None, 0.015392958914552885,
        0.0049995129722962585, 0.45681084201316247, None,
        0.0142010047035309, None, 0.0070595689633190005, None,
        0.005952845854844551, None, 0.4061492485774763, None, None,
        0.006494014427104048, None, 0.011703623854212647, None, None, None,
        None, 0.07835436012162941, None, None, 0.034821718003819786, None,
        None, 0.01206364796632641, None, 0.03109327499333218,
        0.039787780364324274, 0.03686711696251255, None,
        0.26161512904381273, 0.010658776082839268, None,
        0.015031715969771863, 0.009111839906903289, 0.013567279678462564,
        None, 0.011979312007206802, 0.015544303824269074,

In [239]:
sum_rmsds = 0.0
for array in total_rmsd: 
    array[array == None] = 0
    sum_rmsds += array


In [240]:
sum_rmsds

array([0.0, 0.02643773041194726, 0.0, 0.7203649549722806, 0.0, 0.0,
       0.044585819921669705, 0.39667002346553304, 0.11042664148659413,
       0.09356735287336784, 0.09600699053234536, 0.0, 0.0,
       0.010402139360240266, 0.0, 0.11154830690513085, 0.0,
       0.040114166446245644, 0.0, 0.0, 0.08790754012457501, 0.0,
       0.12586858345217686, 0.18978373108704039, 0.0, 0.07027378598187721,
       0.28491960318695214, 0.028024114691593056, 0.0,
       0.030597011627749315, 0.15901866894297573, 0.07725411565407254,
       1.9073571848934634, 0.060560893089742676, 0.1382241957235572,
       0.6927905512685866, 0.080089893058446, 0.017212908255501043,
       0.0752625046572284, 0.0, 0.4061492485774763, 0.04072526779985386,
       0.0, 0.08951222231478, 0.0, 0.15921098674368178, 0.0, 0.0, 0.0,
       0.0, 0.7940599859266587, 0.0, 0.0, 0.27181322628468907, 0.0, 0.0,
       0.10651471182872774, 0.0, 0.3316867011955834, 0.4177382270337844,
       0.29211525402579525, 0.0, 1.91066571377879

In [241]:
match_rate = sum(sum_rmsds > 0) / len(sum_rmsds)

In [246]:
match_rate = sum(total_rmsd[0] > 0) / len(total_rmsd[0])

In [247]:
match_rate

0.39